In [2]:
from sqlite3.dbapi2 import paramstyle

from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from tqdm import tqdm
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector
import warnings
warnings.filterwarnings('ignore')






## Using k fold with logistic regression

In [3]:
wisconsin = pd.read_csv("BreastCancer.csv", index_col=0)
y = wisconsin['Class']
X = wisconsin.drop('Class', axis=1)

In [4]:
lr = LogisticRegression()
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)
results = cross_val_score(lr, X, y, cv=kfold) #default to accuracy
results


array([0.95      , 0.96428571, 0.96428571, 0.97142857, 0.97122302])

In [5]:
print(results.mean())

0.9642446043165467


In [6]:
results = cross_val_score(lr, X, y, cv=kfold, scoring='roc_auc')
print(results.mean())

0.9944643791582568


In [7]:
results = cross_val_score(lr, X, y, cv=kfold, scoring='neg_log_loss')
print(results.mean())

-0.09878376446090598


## using solver with logistic regression

In [8]:
solvers = ['lbfgs','liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
Cs = np.linspace(0.001, 4, 20)
scores = []
for s in solvers:
    for C in Cs:
        lr = LogisticRegression(solver = s, C = C)
        results = cross_val_score(lr, X, y, cv=kfold, scoring='roc_auc')
        scores.append([s,C, results.mean()])
df_scores = pd.DataFrame(scores, columns=['solver','C', 'score'])
df_scores.sort_values('score', ascending=False)


,solver,C,score
1,lbfgs,0.211474,0.994555
41,newton-cg,0.211474,0.994555
61,newton-cholesky,0.211474,0.994555
33,liblinear,2.737158,0.994554
40,newton-cg,0.001000,0.994553
...,...,...,...
113,saga,2.737158,0.993648
119,saga,4.000000,0.993648
22,liblinear,0.421947,0.993376
21,liblinear,0.211474,0.991343


## using full grid search with solver

In [9]:
from sklearn.model_selection import GridSearchCV

In [10]:
params = {'solver': ['lbfgs','liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
          'C': np.linspace(0.001, 4, 20)}
gcv = GridSearchCV(lr, param_grid=params, cv=kfold, scoring='roc_auc')
gcv.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...solver='saga')
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': array([1.0000...00000000e+00]), 'solver': ['lbfgs', 'liblinear', ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: 

In [11]:
print(gcv.best_params_)
print(gcv.best_score_)

{'C': np.float64(0.2114736842105263), 'solver': 'lbfgs'}
0.9945549588684017


In [12]:
df_cv = pd.DataFrame( gcv.cv_results_)
df_cv

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.009653,2.654328e-03,0.003204,0.000393,0.001,lbfgs,"{'C': 0.001, 'solver': 'lbfgs'}",0.992754,0.995697,0.991848,0.998879,0.993590,0.994553,0.002510,5
1,0.002987,1.184017e-05,0.002996,0.000012,0.001,liblinear,"{'C': 0.001, 'solver': 'liblinear'}",0.935236,0.947917,0.978487,0.953353,0.978251,0.958649,0.017142,120
2,0.005982,2.129279e-06,0.002988,0.000013,0.001,newton-cg,"{'C': 0.001, 'solver': 'newton-cg'}",0.992754,0.995697,0.991848,0.998879,0.993590,0.994553,0.002510,5
3,0.003395,4.858384e-04,0.003193,0.000391,0.001,newton-cholesky,"{'C': 0.001, 'solver': 'newton-cholesky'}",0.992754,0.995697,0.991848,0.998879,0.993590,0.994553,0.002510,5
4,0.005986,7.136645e-07,0.002904,0.000173,0.001,sag,"{'C': 0.001, 'solver': 'sag'}",0.992754,0.995471,0.991848,0.998879,0.993590,0.994508,0.002491,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,0.002985,1.150157e-05,0.003004,0.000013,4.000,liblinear,"{'C': 4.0, 'solver': 'liblinear'}",0.992527,0.996377,0.990263,0.999103,0.994277,0.994509,0.003053,8
116,0.007985,6.315872e-04,0.003186,0.000389,4.000,newton-cg,"{'C': 4.0, 'solver': 'newton-cg'}",0.992754,0.996830,0.989130,0.998879,0.994505,0.994420,0.003360,35
117,0.003991,9.615572e-06,0.003384,0.000493,4.000,newton-cholesky,"{'C': 4.0, 'solver': 'newton-cholesky'}",0.992754,0.996830,0.989130,0.998879,0.994505,0.994420,0.003360,35
118,0.006988,1.139990e-05,0.003191,0.000385,4.000,sag,"{'C': 4.0, 'solver': 'sag'}",0.991621,0.995697,0.990489,0.998654,0.994048,0.994102,0.002915,93


## using RandomizedSearch CV

In [13]:
params = {'solver': ['lbfgs','liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
          'C': np.linspace(0.001, 4, 20)}
rgcv = RandomizedSearchCV(lr, param_distributions=params, cv=kfold, scoring='roc_auc', n_iter=20, random_state=25, verbose=3)
rgcv.fit(X, y)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.993 total time=   0.0s
[CV 2/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.997 total time=   0.0s
[CV 3/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.989 total time=   0.0s
[CV 4/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.999 total time=   0.0s
[CV 5/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.994 total time=   0.0s
[CV 1/5] END .C=1.2638421052631577, solver=saga;, score=0.989 total time=   0.0s
[CV 2/5] END .C=1.2638421052631577, solver=saga;, score=0.995 total time=   0.0s
[CV 3/5] END .C=1.2638421052631577, solver=saga;, score=0.994 total time=   0.0s
[CV 4/5] END .C=1.2638421052631577, solver=saga;, score=0.998 total time=   0.0s
[CV 5/5] END .C=1.2638421052631577, solver=saga;, score=0.993 total time=   0.0s
[CV 1/5] END .C=2.5266842105263154, solver=saga;, score=0.988 total time=   0.

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...solver='saga')
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'C': array([1.0000...00000000e+00]), 'solver': ['lbfgs', 'liblinear', ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",3
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",25
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best

In [14]:
print(rgcv.best_params_)
print(rgcv.best_score_)

{'solver': 'lbfgs', 'C': np.float64(0.2114736842105263)}
0.9945549588684017


In [15]:
df_rcv = pd.DataFrame( rgcv.cv_results_)
df_rcv

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_solver,param_C,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.004383,0.000482,0.002992,4.101908e-07,newton-cholesky,0.632421,"{'solver': 'newton-cholesky', 'C': 0.632421052...",0.992754,0.996603,0.989357,0.998879,0.994277,0.994374,0.003258,12
1,0.007984,0.000012,0.002994,4.607343e-06,saga,1.263842,"{'solver': 'saga', 'C': 1.2638421052631577}",0.988678,0.995018,0.993659,0.997982,0.993132,0.993694,0.003021,18
2,0.007891,0.000173,0.002988,1.455174e-05,saga,2.526684,"{'solver': 'saga', 'C': 2.5266842105263154}",0.988451,0.995018,0.993659,0.997982,0.993132,0.993648,0.003096,19
3,0.006977,0.000012,0.003195,3.963820e-04,sag,1.053368,"{'solver': 'sag', 'C': 1.0533684210526315}",0.991848,0.995697,0.990716,0.998879,0.994048,0.994237,0.002892,13
4,0.004593,0.000493,0.003186,4.011276e-04,newton-cholesky,1.053368,"{'solver': 'newton-cholesky', 'C': 1.053368421...",0.992980,0.996830,0.989130,0.998879,0.994277,0.994419,0.003339,9
5,0.003995,0.000011,0.002992,7.835234e-07,newton-cholesky,0.842895,"{'solver': 'newton-cholesky', 'C': 0.842894736...",0.992980,0.996830,0.989130,0.998879,0.994277,0.994419,0.003339,9
6,0.006785,0.000396,0.003182,3.980988e-04,sag,2.526684,"{'solver': 'sag', 'C': 2.5266842105263154}",0.991848,0.995697,0.990489,0.998654,0.994048,0.994147,0.002877,15
7,0.010791,0.000425,0.003004,1.434844e-05,lbfgs,0.211474,"{'solver': 'lbfgs', 'C': 0.2114736842105263}",0.993207,0.996603,0.989810,0.998879,0.994277,0.994555,0.003076,1
8,0.008178,0.000739,0.003198,4.089209e-04,newton-cg,3.789526,"{'solver': 'newton-cg', 'C': 3.7895263157894736}",0.992754,0.996830,0.989130,0.998879,0.994505,0.994420,0.003360,6
9,0.007780,0.000399,0.002991,1.215701e-06,newton-cg,0.632421,"{'solver': 'newton-cg', 'C': 0.6324210526315789}",0.992980,0.996603,0.989357,0.998879,0.994277,0.994419,0.003237,9
